# Validation checks for the Pig cleaning step

Reproduces the trip-cleaning rules from `a2.pig` in pandas on a local copy of `Trips.txt`, to check how many records the Pig script should keep.


In [2]:
import pandas as pd

In [3]:
names = ["trip_id", "taxi_id", "company_id", "dropoff_lat", "dropoff_lon", "distance_km", "fare"]

In [4]:
trips = pd.read_csv('Trips.txt', sep='\t', header=None, names=names)

In [ ]:
trips

In [6]:
trips.isna().sum()

trip_id        0
taxi_id        0
company_id     4
dropoff_lat    2
dropoff_lon    4
distance_km    3
fare           1
dtype: int64

In [8]:
trips.describe

<bound method NDFrame.describe of     trip_id  taxi_id  company_id  dropoff_lat  dropoff_lon  distance_km   fare
0        47      100         5.0     37.75510   -122.48105        12.39  38.84
1        86      100         2.0     37.74983   -122.43255        11.08  27.23
2        69      100         3.0     37.72532   -122.49350        10.36  31.80
3        12      101         4.0     37.78995   -122.42200        21.99  65.92
4        91      101         1.0     37.71841          NaN        18.09  45.48
..      ...      ...         ...          ...          ...          ...    ...
95       42      117         5.0     37.75706   -122.42996        -3.15   7.11
96        3      118         3.0     37.75252   -122.41249        18.51  46.05
97       57      118         2.0     37.70851   -122.49581        16.29  51.47
98       77      118         2.0     37.78873   -122.42503        10.91  28.52
99       26      119         4.0     37.76573   -122.46496        14.17  33.03

[100 rows x 7 col

### Task 1 Sanity Checks

In [ ]:
df_with_missing = trips[trips.isnull().any(axis=1)]
print(df_with_missing)


In [13]:
df_with_missing.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10 entries, 4 to 82
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   trip_id      10 non-null     int64  
 1   taxi_id      10 non-null     int64  
 2   company_id   6 non-null      float64
 3   dropoff_lat  8 non-null      float64
 4   dropoff_lon  6 non-null      float64
 5   distance_km  7 non-null      float64
 6   fare         9 non-null      float64
dtypes: float64(5), int64(2)
memory usage: 640.0 bytes


In [ ]:
# Remove rows with any missing values
trips_cl1 = trips.dropna()

In [ ]:
# 90 records left after removing records w/ NaN values
trips_cl1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 90 entries, 0 to 99
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   trip_id      90 non-null     int64  
 1   taxi_id      90 non-null     int64  
 2   company_id   90 non-null     float64
 3   dropoff_lat  90 non-null     float64
 4   dropoff_lon  90 non-null     float64
 5   distance_km  90 non-null     float64
 6   fare         90 non-null     float64
dtypes: float64(5), int64(2)
memory usage: 5.6 KB


In [ ]:
# Filter out negative or zero distances
neg_dist_records =  trips_cl1[trips_cl1['distance_km'] <= 0]

In [ ]:
# One record found with negative distance
neg_dist_records.head()

In [35]:
neg_dist_records.shape

(1, 7)

In [36]:
num_removed_records = 1

In [ ]:
# Filter out distances greater than 20 km as outliers
outlier_dis_records = trips_cl1[trips_cl1['distance_km'] > 20]

In [ ]:
# 18 records found with distance greater than 20 km
outlier_dis_records.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18 entries, 3 to 92
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   trip_id      18 non-null     int64  
 1   taxi_id      18 non-null     int64  
 2   company_id   18 non-null     float64
 3   dropoff_lat  18 non-null     float64
 4   dropoff_lon  18 non-null     float64
 5   distance_km  18 non-null     float64
 6   fare         18 non-null     float64
dtypes: float64(5), int64(2)
memory usage: 1.1 KB


In [38]:
num_removed_records += 18

In [40]:
# Filter out fares less than $5 as outliers
outlier_fare_records = trips_cl1[trips_cl1['fare'] < 5]

In [ ]:
# 3 records found < 5 dollars
outlier_fare_records.head()

In [42]:
outlier_fare_records.shape

(3, 7)

In [43]:
num_removed_records += 3

In [44]:
num_removed_records

22

In [ ]:
filtered_trips = trips_cl1[(trips_cl1['distance_km'] > 0) & 
                           (trips_cl1['distance_km'] <= 20) & 
                           (trips_cl1['fare'] >= 5)]

In [ ]:
filtered_trips.head()

In [28]:
filtered_trips.info()

<class 'pandas.core.frame.DataFrame'>
Index: 68 entries, 0 to 99
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   trip_id      68 non-null     int64  
 1   taxi_id      68 non-null     int64  
 2   company_id   68 non-null     float64
 3   dropoff_lat  68 non-null     float64
 4   dropoff_lon  68 non-null     float64
 5   distance_km  68 non-null     float64
 6   fare         68 non-null     float64
dtypes: float64(5), int64(2)
memory usage: 4.2 KB


In [ ]:
print(filtered_trips)